# [2026-08-01-01] Clinical Trials by Highest Phase

Self-contained notebook that compares the number of **interventional studies registered on
ClinicalTrials.gov** for ME/CFS and CrunchME's adult/all-ages comparator diseases.

Trials are shown as a stacked horizontal bar chart by their **highest registered phase**.
Studies without an FDA-defined phase (common for behavioural, device and other non-drug
interventions) are retained as `Not applicable / unreported`. Run all cells top to bottom;
the final cells render the CrunchME-style chart and export the source data and image.

**Default scope:** ME/CFS plus the adult core comparator groups from CrunchME's comparator
framework. Optional adult and Long COVID comparator groups can be enabled in section 2.2.


### 📦 1 | Install & Import


#### 1.1 | Install packages


In [ ]:
# Colab already includes most of these packages. Install only anything that is missing.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "requests": "requests",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "plotnine": "plotnine",
    "mizani": "mizani",
}

for package, module in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )


#### 1.2 | Import packages


In [ ]:
from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import time

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import requests
from mizani.formatters import comma_format
from plotnine import (
    aes,
    coord_flip,
    element_blank,
    element_line,
    element_rect,
    element_text,
    geom_col,
    geom_text,
    ggplot,
    guide_legend,
    guides,
    labs,
    scale_fill_manual,
    scale_y_continuous,
    theme,
    theme_classic,
)


#### 1.3 | Load CrunchME brand font (Brutel)

Searches common Colab/Drive locations for `Brutel*.otf` or `Brutel*.ttf`. If none are
found, Colab offers an upload prompt. The chart still renders with a sans-serif fallback.


In [ ]:
BRAND_FONT = "Brutel"

FONT_SEARCH_DIRS = [
    ".",
    "/content",
    "/content/fonts",
    "/content/drive/MyDrive/CrunchME/Fonts",
    "/content/drive/MyDrive/CrunchME Visual Branding/Font/brutel",
]

def _find_brutel(dirs):
    import glob
    import os

    hits = []
    for directory in dirs:
        for extension in ("otf", "ttf"):
            hits += glob.glob(
                os.path.join(directory, "**", f"Brutel*.{extension}"),
                recursive=True,
            )
    return sorted(set(hits))

font_files = _find_brutel(FONT_SEARCH_DIRS)

if not font_files:
    try:
        from google.colab import files

        print(
            "No Brutel files found. Upload Brutel-Regular and Brutel-Bold "
            "(other weights are optional):"
        )
        files.upload()
        font_files = _find_brutel(["."])
    except Exception as exc:
        print("Automatic font upload unavailable:", exc)

for font_path in font_files:
    try:
        fm.fontManager.addfont(font_path)
    except Exception as exc:
        print("Could not register", font_path, "-", exc)

available_fonts = {font.name for font in fm.fontManager.ttflist}
if BRAND_FONT in available_fonts:
    plt.rcParams["font.family"] = BRAND_FONT
    print(
        f"✓ Brand font '{BRAND_FONT}' active "
        f"({len(font_files)} file(s) newly registered)."
    )
else:
    BRAND_FONT = "DejaVu Sans"
    plt.rcParams["font.family"] = BRAND_FONT
    print("⚠ Brutel not found - using DejaVu Sans.")


### 🔧 2 | Parameters


#### 2.1 | Comparator sets


In [ ]:
# Comparator groups transcribed from CrunchME's 1 August 2026 framework.
# The two ME/CFS core groups are used by default.
ME_CFS_CORE_LOWER = [
    "Parkinson's disease",
    "Multiple sclerosis",
    "Inflammatory bowel disease (IBD)",
    "Rheumatoid arthritis",
    "Lupus",
]

ME_CFS_CORE_HIGHER = [
    "Epilepsy",
    "Asthma",
    "Type 1 diabetes",
]

ME_CFS_OPTIONAL = [
    "HIV/AIDS",
    "Breast cancer",
    "Prostate cancer",
    "Migraine",
    "Endometriosis",
    "ALS",
]

LONG_COVID_SET = [
    "Long COVID",
    "Chronic obstructive pulmonary disease (COPD)",
    "Chronic kidney disease",
    "Type 2 diabetes",
    "Stroke",
]


#### 2.2 | Scope switches


In [ ]:
# Core adult ME/CFS comparators are always included.
INCLUDE_OPTIONAL_ME_COMPARATORS = False
INCLUDE_LONG_COVID_SET = False
EXCLUDE_DISEASES = ["Asthma"]

# Add or override diseases here if needed: {"Display name": 'ClinicalTrials.gov query'}
CUSTOM_DISEASES = {}

disease_list = (
    ["ME/CFS"]
    + ME_CFS_CORE_LOWER
    + ME_CFS_CORE_HIGHER
)
if INCLUDE_OPTIONAL_ME_COMPARATORS:
    disease_list += ME_CFS_OPTIONAL
if INCLUDE_LONG_COVID_SET:
    disease_list += LONG_COVID_SET
disease_list += list(CUSTOM_DISEASES)

# Preserve order, remove duplicates and apply any chart exclusions.
disease_list = [
    disease for disease in dict.fromkeys(disease_list)
    if disease not in EXCLUDE_DISEASES
]
print(f"{len(disease_list)} diseases selected:")
print("\n".join(f"  - {disease}" for disease in disease_list))


#### 2.3 | Search definitions


In [ ]:
# query.cond searches the ClinicalTrials.gov condition search area, including submitted
# conditions, condition MeSH terms, titles and keywords. Quoted phrases reduce noise.
CONDITION_QUERIES = {
    "ME/CFS": (
        '"Myalgic Encephalomyelitis" OR "Chronic Fatigue Syndrome" '
        'OR "ME/CFS" OR "Systemic Exertion Intolerance Disease"'
    ),
    "Parkinson's disease": '"Parkinson Disease" OR "Parkinson\'s Disease"',
    "Multiple sclerosis": '"Multiple Sclerosis"',
    "Inflammatory bowel disease (IBD)": (
        '"Inflammatory Bowel Disease" OR "Crohn Disease" '
        'OR "Crohn\'s Disease" OR "Ulcerative Colitis"'
    ),
    "Rheumatoid arthritis": '"Rheumatoid Arthritis"',
    "Lupus": '"Systemic Lupus Erythematosus" OR "Lupus"',
    "HIV/AIDS": (
        '"HIV Infections" OR "Human Immunodeficiency Virus" '
        'OR "Acquired Immunodeficiency Syndrome" OR "HIV/AIDS"'
    ),
    "Breast cancer": '"Breast Cancer" OR "Breast Neoplasms"',
    "Prostate cancer": '"Prostate Cancer" OR "Prostatic Neoplasms"',
    "Migraine": '"Migraine" OR "Migraine Disorders"',
    "Endometriosis": '"Endometriosis"',
    "ALS": '"Amyotrophic Lateral Sclerosis" OR "Motor Neuron Disease"',
    "Epilepsy": '"Epilepsy"',
    "Asthma": '"Asthma"',
    "Type 1 diabetes": (
        '"Type 1 Diabetes" OR "Type 1 Diabetes Mellitus" '
        'OR "Diabetes Mellitus, Type 1"'
    ),
    "Long COVID": (
        '"Long COVID" OR "Post-COVID Conditions" OR "Post-Acute Sequelae of SARS-CoV-2" '
        'OR "Post-Acute COVID-19 Syndrome"'
    ),
    "Chronic obstructive pulmonary disease (COPD)": (
        '"Chronic Obstructive Pulmonary Disease" OR "COPD"'
    ),
    "Chronic kidney disease": '"Chronic Kidney Disease"',
    "Type 2 diabetes": (
        '"Type 2 Diabetes" OR "Type 2 Diabetes Mellitus" '
        'OR "Diabetes Mellitus, Type 2"'
    ),
    "Stroke": '"Stroke" OR "Cerebrovascular Accident"',
}

CONDITION_QUERIES.update(CUSTOM_DISEASES)

missing_queries = [d for d in disease_list if d not in CONDITION_QUERIES]
if missing_queries:
    raise KeyError(f"Missing condition queries for: {missing_queries}")

selected_queries = {d: CONDITION_QUERIES[d] for d in disease_list}
selected_queries


#### 2.4 | Phase groups, colours and outputs


In [ ]:
# Each interventional study is assigned to one mutually exclusive bucket.
# For combined phases (e.g. Phase 1/Phase 2), the highest phase is used.
PHASE_FILTERS = OrderedDict(
    [
        ("Phase 4", "AREA[Phase]PHASE4"),
        (
            "Phase 3",
            "AREA[Phase]PHASE3 AND NOT AREA[Phase]PHASE4",
        ),
        (
            "Phase 2",
            "AREA[Phase]PHASE2 AND NOT AREA[Phase]PHASE3 "
            "AND NOT AREA[Phase]PHASE4",
        ),
        (
            "Early / Phase 1",
            "(AREA[Phase]EARLY_PHASE1 OR AREA[Phase]PHASE1) "
            "AND NOT AREA[Phase]PHASE2 AND NOT AREA[Phase]PHASE3 "
            "AND NOT AREA[Phase]PHASE4",
        ),
    ]
)

# Legend and stacking order.
PHASE_ORDER = [
    "Early / Phase 1",
    "Phase 2",
    "Phase 3",
    "Phase 4",
    "Not applicable / unreported",
]

# Phase colours use the CrunchME disease palette; grey remains neutral for
# studies whose phase is not applicable or was not reported.
PHASE_COLORS = {
    "Not applicable / unreported": "#495057",
    "Early / Phase 1": "#7BB3F3",  # ME/CFS blue
    "Phase 2": "#D7B3FF",         # Dysautonomia lavender
    "Phase 3": "#E57399",         # Long COVID pink
    "Phase 4": "#FFD8A8",         # Fibromyalgia peach
}

API_BASE = "https://clinicaltrials.gov/api/v2"
USE_CACHE = True
OUTPUT_CSV = "clinical_trials_me_comparators.csv"
OUTPUT_PNG = "clinical_trials_me_comparators.png"


### 🚰 3 | Data Import - ClinicalTrials.gov API v2


#### 3.1 | API helper

The API is queried for counts only (`pageSize=1`, `countTotal=true`), rather than
downloading tens of thousands of full records. Requests use retry/backoff for transient
errors. ClinicalTrials.gov's dataset version timestamp is saved with every result.


In [ ]:
session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "CrunchME-clinical-trials-analysis/1.0 "
            "(https://www.crunchme.org/)"
        )
    }
)

def get_json(url, params=None, attempts=4, timeout=45):
    for attempt in range(attempts):
        try:
            response = session.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            if attempt == attempts - 1:
                raise
            wait_seconds = 1.5 * (2 ** attempt)
            print(f"  API retry in {wait_seconds:.1f}s: {exc}")
            time.sleep(wait_seconds)

def clinicaltrials_count(condition_query, extra_filter=None):
    advanced_filter = "AREA[StudyType]INTERVENTIONAL"
    if extra_filter:
        advanced_filter += f" AND ({extra_filter})"

    payload = get_json(
        f"{API_BASE}/studies",
        params={
            "format": "json",
            "query.cond": condition_query,
            "filter.advanced": advanced_filter,
            "pageSize": 1,
            "countTotal": "true",
            "fields": "NCTId",
        },
    )
    return int(payload.get("totalCount", 0))

try:
    api_version_info = get_json(f"{API_BASE}/version")
    DATA_TIMESTAMP = api_version_info.get("dataTimestamp", "unknown")
    API_VERSION = api_version_info.get("apiVersion", "unknown")
except Exception as exc:
    print("Could not retrieve API version metadata:", exc)
    DATA_TIMESTAMP = "unknown"
    API_VERSION = "unknown"

print(f"ClinicalTrials.gov API version: {API_VERSION}")
print(f"Dataset timestamp: {DATA_TIMESTAMP}")


#### 3.2 | Fetch counts


In [ ]:
cache_signature = hashlib.sha256(
    json.dumps(
        {
            "queries": selected_queries,
            "phase_filters": list(PHASE_FILTERS.items()),
            "study_type": "INTERVENTIONAL",
            "data_timestamp": DATA_TIMESTAMP,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]
CACHE_FILE = f"clinical_trials_cache_{cache_signature}.csv"

if USE_CACHE:
    try:
        cached_df = pd.read_csv(CACHE_FILE)
        expected_columns = {
            "Disease", "Stage", "Trials", "Condition query",
            "API version", "Data timestamp", "Retrieved at (UTC)",
        }
        if not expected_columns.issubset(cached_df.columns):
            raise ValueError("Cache columns do not match this notebook version.")
        trial_counts_df = cached_df
        print(f"✓ Loaded cached data from {CACHE_FILE}")
    except (FileNotFoundError, ValueError):
        trial_counts_df = None
else:
    trial_counts_df = None

if trial_counts_df is None:
    rows = []
    retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

    for disease, condition_query in selected_queries.items():
        print(f"Fetching {disease} ...")
        total = clinicaltrials_count(condition_query)
        phased_total = 0
        disease_rows = []

        for stage, phase_filter in PHASE_FILTERS.items():
            count = clinicaltrials_count(condition_query, phase_filter)
            phased_total += count
            disease_rows.append((stage, count))
            time.sleep(0.08)

        residual = total - phased_total
        if residual < 0:
            raise ValueError(
                f"Phase counts exceed total for {disease}: "
                f"{phased_total} > {total}. Check API query logic."
            )

        disease_rows.append(("Not applicable / unreported", residual))
        for stage, count in disease_rows:
            rows.append(
                {
                    "Disease": disease,
                    "Stage": stage,
                    "Trials": count,
                    "Condition query": condition_query,
                    "API version": API_VERSION,
                    "Data timestamp": DATA_TIMESTAMP,
                    "Retrieved at (UTC)": retrieved_at,
                }
            )

    trial_counts_df = pd.DataFrame(rows)
    if USE_CACHE:
        trial_counts_df.to_csv(CACHE_FILE, index=False)
        print(f"✓ Cached data to {CACHE_FILE}")

trial_counts_df.head(10)


### 🥋 4 | Data Wrangling & Checks


#### 4.1 | Order diseases and validate totals


In [ ]:
# Keep only the diseases selected in section 2.2. This also makes old cache files safe
# to reuse after changing EXCLUDE_DISEASES.
trial_counts_df = trial_counts_df[
    trial_counts_df["Disease"].isin(disease_list)
].copy()
trial_counts_df["Trials"] = trial_counts_df["Trials"].astype(int)
# plotnine stacks categorical groups in reverse order. Reversing the underlying
# categories makes the bars read left-to-right in PHASE_ORDER, while the legend
# is explicitly kept in PHASE_ORDER below.
trial_counts_df["Stage"] = pd.Categorical(
    trial_counts_df["Stage"],
    categories=list(reversed(PHASE_ORDER)),
    ordered=True,
)

totals_df = (
    trial_counts_df.groupby("Disease", observed=True, as_index=False)["Trials"]
    .sum()
    .rename(columns={"Trials": "Total"})
    .sort_values("Total", ascending=True)
)
disease_order = totals_df["Disease"].tolist()

trial_counts_df["Disease"] = pd.Categorical(
    trial_counts_df["Disease"],
    categories=disease_order,
    ordered=True,
)
trial_counts_df = trial_counts_df.sort_values(["Disease", "Stage"])

# One row per selected disease and phase bucket, and all counts are non-negative.
expected_rows = len(disease_list) * len(PHASE_ORDER)
assert len(trial_counts_df) == expected_rows, (
    f"Expected {expected_rows} rows, found {len(trial_counts_df)}"
)
assert (trial_counts_df["Trials"] >= 0).all()
assert set(totals_df["Disease"].astype(str)) == set(disease_list)

totals_df["Total label"] = totals_df["Total"].map(lambda value: f"{value:,}")
totals_df


#### 4.2 | Exact counts table


In [ ]:
counts_table = (
    trial_counts_df.pivot_table(
        index="Disease",
        columns="Stage",
        values="Trials",
        aggfunc="sum",
        fill_value=0,
        observed=True,
    )
    .reindex(index=list(reversed(disease_order)), columns=PHASE_ORDER)
)
counts_table["Total"] = counts_table.sum(axis=1)
counts_table


### 📊 5 | Chart - Registered Interventional Studies by Highest Phase


In [ ]:
background = "#121421"
white = "#FFFFFF"
muted = "#ADB5BD"

data_date = (
    DATA_TIMESTAMP[:10]
    if isinstance(DATA_TIMESTAMP, str) and DATA_TIMESTAMP != "unknown"
    else "latest available"
)

max_total = max(int(totals_df["Total"].max()), 1)
totals_df["Label position"] = totals_df["Total"] + max_total * 0.012

chart_height = max(8.0, 0.72 * len(disease_order) + 3.2)

plot_trials = (
    ggplot(
        trial_counts_df,
        aes(x="Disease", y="Trials", fill="Stage"),
    )
    + geom_col(width=0.78)
    + geom_text(
        data=totals_df,
        mapping=aes(
            x="Disease",
            y="Label position",
            label="Total label",
        ),
        inherit_aes=False,
        ha="left",
        color=white,
        size=17,
        family=BRAND_FONT,
    )
    + coord_flip()
    + labs(
        title="Number of registered clinical trials for ME/CFS vs. other select diseases",
        x="",
        y="Number of registered interventional studies",
        fill="Highest phase",
    )
    + theme_classic()
    + theme(
        figure_size=(13, chart_height),
        text=element_text(size=14, family=BRAND_FONT),
        plot_title_position="plot",
        plot_margin_left=0.025,
        plot_margin_right=0.025,
        plot_margin_top=0.018,
        plot_margin_bottom=0.10,
        plot_title=element_text(
            size=26,
            weight="bold",
            color=white,
            ha="left",
            family=BRAND_FONT,
        ),
        axis_title_x=element_text(
            size=19,
            weight="bold",
            color=white,
            margin={"t": 14, "units": "pt"},
            family=BRAND_FONT,
        ),
        axis_text_x=element_text(
            size=18,
            color=white,
            family=BRAND_FONT,
        ),
        axis_text_y=element_text(
            size=17,
            color=white,
            margin={"r": 10, "units": "pt"},
            family=BRAND_FONT,
        ),
        legend_title=element_text(
            size=20,
            weight="bold",
            color=white,
            ha="left",
            margin={"b": 16, "units": "pt"},
            family=BRAND_FONT,
        ),
        legend_text=element_text(
            size=19,
            color=white,
            family=BRAND_FONT,
        ),
        legend_position=(-0.43, -0.245),
        legend_direction="horizontal",
        legend_title_position="top",
        legend_justification="left",
        legend_box_just="left",
        plot_background=element_rect(fill=background, color=background, size=0),
        panel_background=element_rect(fill=background, color=background, size=0),
        legend_background=element_rect(fill=background, color=background, size=0),
        legend_key=element_rect(fill=background, color=background, size=0),
        axis_line=element_line(color=white, size=0.8),
        axis_ticks=element_line(color=white, size=0.8),
        panel_grid_major_x=element_line(
            color=white,
            size=0.45,
            alpha=0.22,
        ),
        panel_grid_major_y=element_blank(),
        panel_grid_minor=element_blank(),
    )
    + scale_y_continuous(
        labels=comma_format(),
        expand=(0, 0, 0.10, 0),
    )
    + scale_fill_manual(
        values=PHASE_COLORS,
        breaks=PHASE_ORDER,
    )
)

plot_trials


### 💾 6 | Export


In [ ]:
# Export exact counts and a high-resolution visual.
export_df = trial_counts_df.copy()
export_df["Disease"] = export_df["Disease"].astype(str)
export_df["Stage"] = export_df["Stage"].astype(str)
export_df.to_csv(OUTPUT_CSV, index=False)
plot_trials.save(
    OUTPUT_PNG,
    width=13,
    height=chart_height,
    dpi=300,
    facecolor=background,
    edgecolor=background,
    verbose=False,
)

print(f"✓ Saved data:  {OUTPUT_CSV}")
print(f"✓ Saved chart: {OUTPUT_PNG}")

# Uncomment in Colab to download both files:
# from google.colab import files
# files.download(OUTPUT_CSV)
# files.download(OUTPUT_PNG)


### Methodology and interpretation notes

- **Coverage:** This is a comparison of registrations on ClinicalTrials.gov, not a complete
  census of every trial conducted worldwide. Registration rules and historical coverage
  vary by country, sponsor, intervention type and time period.
- **Study type:** Only records classified as `INTERVENTIONAL` are counted. Observational
  studies and expanded-access records are excluded.
- **Phase assignment:** Combined-phase studies are assigned to their highest phase, so the
  stacked segments sum to the true disease total. Phase is often not applicable to devices,
  behavioural interventions and other non-drug studies; these remain in the grey segment.
- **Condition matching:** Queries use the ClinicalTrials.gov condition search area and
  disease synonyms shown in section 2.3. A study covering multiple selected conditions can
  legitimately appear in more than one disease bar.
- **Reproducibility:** The API version, source-data timestamp, retrieval time and exact
  condition query are included in the exported CSV. Delete the generated cache file to
  force a fresh API pull.
